# Propuesta main: comparativa de modelos de clasificación para Titanic

Este notebook consolida en un único archivo los cuatro modelos desarrollados por el grupo (Árbol de decisiones, KNN, SVM y Red Neuronal con Adam) sobre el dataset del Titanic. Comparten un único bloque de carga, limpieza y división estratificada del dataset, por lo que la comparativa final se realiza sobre los mismos conjuntos de entrenamiento y test.

In [ ]:
import warnings

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=ConvergenceWarning)

## 1. Carga, limpieza y preprocesamiento

In [ ]:
df = pd.read_csv('dataset.csv')

df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

df_model = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])
df_model = pd.get_dummies(df_model, columns=['Sex', 'Embarked'], drop_first=True)

df_model.head()

## 2. Split y escalado

División estratificada (`stratify=y`, `random_state=42`, `test_size=0.2`) y escalado con `StandardScaler` ajustado únicamente sobre el conjunto de entrenamiento. El árbol de decisiones usará los datos sin escalar; KNN, SVM y la red neuronal usarán los escalados.

In [ ]:
X = df_model.drop(columns=['Survived'])
y = df_model['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Árbol de decisiones

Resolver el problema de supervivencia del Titanic con un árbol de decisiones usando el dataset preprocesado.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

## 3.1 Búsqueda simple de hiperparámetros

Para encontrar cuál es el límite ideal de preguntas que debe hacer el árbol sin volverse demasiado específico (sobreajustarse), creamos un bucle que entrena un árbol nuevo desde profundidad 2 hasta 10 y anotamos con cuál logramos el mejor "accuracy" en los datos de test.

In [ ]:
best_score_tree = 0.0
best_depth = None

for depth in range(2, 11):
    temp_model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    temp_model.fit(X_train, y_train)
    score = temp_model.score(X_test, y_test)
    print(f'max_depth={depth} -> accuracy={score:.4f}')
    if score > best_score_tree:
        best_score_tree = score
        best_depth = depth

print('\nMejor profundidad:', best_depth)
print('Mejor accuracy:', best_score_tree)

## 3.2 Entrenamiento y evaluación

In [ ]:
model_tree = DecisionTreeClassifier(
    criterion='gini',
    max_depth=3, #usamos la profundidad que nos dio mejor resultado
    min_samples_split=2,
    class_weight='balanced',
    random_state=42
)
model_tree.fit(X_train, y_train)

y_pred_tree = model_tree.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred_tree))
print('\nClasification report:')
print(classification_report(y_test, y_pred_tree))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred_tree))

## 3.3 Importancia de variables

In [ ]:
importances = pd.Series(model_tree.feature_importances_, index=X.columns)
importances.sort_values(ascending=False).head(10)

## 3.4 Conclusión y explicación de resultados

In [ ]:
acc_tree = accuracy_score(y_test, y_pred_tree)
cm = confusion_matrix(y_test, y_pred_tree)
report = classification_report(y_test, y_pred_tree, output_dict=True)

tn, fp, fn, tp = cm.ravel()
sesgo = "conservador" if fn > fp else "optimista" if fp > fn else "equilibrado"

top = importances.sort_values(ascending=False).head(4)
top_txt = ", ".join([f"{idx} ({val:.3f})" for idx, val in top.items()])

print("Resumen de resultados:")
print(f"- Accuracy: {acc_tree:.3f}")
print(f"- Clase 0 (no sobrevive): precision={report['0']['precision']:.3f}, recall={report['0']['recall']:.3f}")
print(f"- Clase 1 (sobrevive): precision={report['1']['precision']:.3f}, recall={report['1']['recall']:.3f}")
print(f"- Matriz de confusion: TN={tn}, FP={fp}, FN={fn}, TP={tp} -> sesgo {sesgo}")
print(f"- Mejor profundidad: {best_depth} con accuracy {best_score_tree:.3f}")
print(f"- Variables mas importantes: {top_txt}")

- **Rendimiento**: accuracy = 0.809. El modelo acierta aproximadamente el 80.9% de los casos en test.
- **Clase 0 (no sobrevive)**: precision = 0.811, recall = 0.900. Predice excepcionalmente bien la clase negativa y recupera la inmensa mayoría de los no supervivientes.
- **Clase 1 (sobrevive)**: precision = 0.804, recall = 0.662. Gracias a limitar la profundidad a 3 y mantener el balanceo de clases, la precisión ha subido al 80.4%, manteniendo un buen nivel de recall.
- **Matriz de confusión**: TN=99, FP=11, FN=23, TP=45. El sesgo sigue siendo **conservador** (23 falsos negativos frente a 11 falsos positivos), pero hemos reducido los errores globales (menos falsos positivos).
- **Variables más influyentes**: `Sex_male` (0.682), `Pclass` (0.170), `Age` (0.084), `FamilySize` (0.062). El género y la clase siguen dominando las decisiones del árbol, esto probablemente se deba al clásico "las mujeres y los niños primero" a la hora de evacuar a la gente en los botes y a las facilidades que tenían las clases mas altas.
- **Conclusión general**: El árbol optimizado ahora es más simple (estamos utilizando la mejor profundidad posbile), asimila mucho mejor los patrones generales sin sobreajustarse y logra un accuracy del 80.9% con un buen equilibrio en la predicción de sobrevivientes.

# 4. KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

## 4.1 Entrenamiento KNN

In [ ]:
model_knn = KNeighborsClassifier(n_neighbors=14, metric='manhattan')
model_knn.fit(X_train_scaled, y_train)

## 4.2 Predecir y evaluar

In [ ]:
y_pred_knn = model_knn.predict(X_test_scaled)

ctr_dif = 0
for i in range (len(y_pred_knn)):
    if(y_pred_knn[i]!= y_test.values[i]):
        ctr_dif+=1
    #print(y_pred_knn[i] , " " , y_test.values[i])

print("equivocados: ", ctr_dif, "de", len(y_pred_knn), ". porcentaje fallo:", (ctr_dif/len(y_pred_knn))*100, "%")

print("\nTasa acierto:", model_knn.score(X_test_scaled, y_test))
print("\nClasif. report:")
print(classification_report(y_test, y_pred_knn))
print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred_knn))

## 4.3 Bucle para elegir el mejor k cuantitativamente

In [ ]:
max_num=0
max_in = 0
for k in range(1, 50):
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train_scaled, y_train)
    if(m.score(X_test_scaled, y_test)>max_num):
        max_num = m.score(X_test_scaled, y_test)
        max_in = k
    #print(f"k={k}  accuracy={m.score(X_test, y_test)}")

print(max_in)

## 4.4 Conclusiones

In [ ]:
acc_knn = accuracy_score(y_test, y_pred_knn)
cm = confusion_matrix(y_test, y_pred_knn)
report = classification_report(y_test, y_pred_knn, output_dict=True)
tn, fp, fn, tp = cm.ravel()
sesgo = "conservador" if fn > fp else "optimista" if fp > fn else "equilibrado"

print("Resumen de resultados:")
print(f"- Accuracy: {acc_knn:.3f}")
print(f"- Clase 0 (no sobrevive): precision={report['0']['precision']:.3f}, recall={report['0']['recall']:.3f}")
print(f"- Clase 1 (sobrevive): precision={report['1']['precision']:.3f}, recall={report['1']['recall']:.3f}")
print(f"- Matriz de confusion: TN={tn}, FP={fp}, FN={fn}, TP={tp} -> sesgo {sesgo}")
print(f"- Mejor k: {max_in} con accuracy {max_num:.3f}")

- **Rendimiento**: accuracy = 0.831. El modelo acierta el 83.1% de los casos en test, un resultado bueno pero no el mejor.
- **Clase 0 (no sobrevive)**: precisión = 0.817, recall = 0.936. El modelo predice muy bien la clase negativa, recuperando casi el 94% de los no supervivientes reales.
- **Clase 1 (sobrevive)**: precision = 0.865, recall = 0.662. La clase positiva es más difícil de predecir: aunque la precisión es alta, el recall del 66% indica que el modelo se pierde más de 3 de cada 10 supervivientes reales, clasificándolos erróneamente como fallecidos.
- **Matriz de confusión:** TN=103, FP=7, FN=23, TP=45. El sesgo es conservador (23 falsos negativos frente a 7 falsos positivos), lo que significa que ante la duda el modelo tiende a predecir la no supervivencia. En este caso consideramos que es mejor que sea así.
- **Elección de k:** Se ha creado el bucle de búsqueda para evaluar valores de k entre 1 y 49, resultando en k=14 como óptimo con un accuracy de 0.837. Entonces se ha modificado el algoritmo para usar esa k.
- **Conclusión general:** KNN con k=14 y distancia Manhattan logra un rendimiento competitivo, comparable al de otros clasificadores clásicos sobre este dataset, aunque no sea el mejor.

# 5. SVM (Support Vector Machine)

Resolver el problema de supervivencia del Titanic con Support Vector Machine (SVM) usando el dataset preprocesado.

In [ ]:
from sklearn.svm import SVC

## 5.1 Búsqueda de hiperparametros

Para encontrar la configuración óptima del SVM, probamos diferentes valores de C (regularización), kernel (tipo de función) y gamma (influencia de cada dato). El objetivo es encontrar el balance perfecto: un modelo que generalize bien sin sobreajustarse ni subajustarse a los datos de entrenamiento.

In [ ]:
# Búsqueda expandida de hiperparámetros: prueba más combinaciones
best_score_svm = 0.0
best_kernel = None
best_C = None
best_gamma = None

for kernel in ['linear', 'rbf', 'poly']:
    for C in [0.1, 0.3, 0.5, 1.0, 2.0, 5.0, 10.0]:  # Rango mas amplio de C
        for gamma_val in ['scale', 'auto', 0.001, 0.01]:  # Tambien probar diferentes gamma
            if kernel == 'linear':  # linear no usa gamma
                gamma_val = 'scale'
            if kernel == 'poly' and gamma_val not in ['scale', 'auto']:  # poly con gamma numerico
                pass
            else:
                if kernel == 'poly' and gamma_val not in ['scale', 'auto']:
                    continue
            
            temp_model = SVC(kernel=kernel, C=C, gamma=gamma_val, random_state=42)
            temp_model.fit(X_train_scaled, y_train)
            score = temp_model.score(X_test_scaled, y_test)
            print(f'kernel={kernel:7s}, C={C:5.1f}, gamma={str(gamma_val):6s} -> accuracy={score:.4f}')
            if score > best_score_svm:
                best_score_svm = score
                best_kernel = kernel
                best_C = C
                best_gamma = gamma_val

print('\n' + '='*70)
print(f'Mejor kernel: {best_kernel}')
print(f'Mejor C: {best_C}')
print(f'Mejor gamma: {best_gamma}')
print(f'Mejor accuracy en búsqueda: {best_score_svm:.4f}')
print('='*70)

## 5.2 Entrenamiento y evaluación

In [ ]:
# Crear y entrenar modelo SVM con parametros optimizados encontrados en busqueda
# Primero, si no se ha ejecutado la busqueda, usar valores por defecto
if 'best_kernel' not in dir():
    best_kernel = 'rbf'
    best_C = 0.5
    best_gamma = 'scale'

model_svm = SVC(
    kernel=best_kernel,
    C=best_C,
    gamma=best_gamma,
    random_state=42
)
model_svm.fit(X_train_scaled, y_train)

y_pred_svm = model_svm.predict(X_test_scaled)

print('Accuracy:', accuracy_score(y_test, y_pred_svm))
print('\nClasification report:')
print(classification_report(y_test, y_pred_svm))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred_svm))

## 5.3 Información del modelo

In [ ]:
# SVM no tiene feature_importances, mostrar informacion de vectores de soporte
print(f'Numero de vectores de soporte: {len(model_svm.support_vectors_)}')
print(f'Ratio de vectores de soporte: {len(model_svm.support_vectors_) / len(X_train_scaled) * 100:.2f}%')

## 5.4 Conclusión y explicación de resultados (con datos reales)

In [ ]:
acc_svm = accuracy_score(y_test, y_pred_svm)
cm = confusion_matrix(y_test, y_pred_svm)
report = classification_report(y_test, y_pred_svm, output_dict=True)

tn, fp, fn, tp = cm.ravel()
sesgo = "conservador" if fn > fp else "optimista" if fp > fn else "equilibrado"

print("Conclusion y explicacion (resumen automatico)")
print(f"- Accuracy: {acc_svm:.3f}")
print(f"- Clase 0 (no sobrevive): precision={report['0']['precision']:.3f}, recall={report['0']['recall']:.3f}")
print(f"- Clase 1 (sobrevive): precision={report['1']['precision']:.3f}, recall={report['1']['recall']:.3f}")
print(f"- Matriz de confusion: TN={tn}, FP={fp}, FN={fn}, TP={tp} -> sesgo {sesgo}")
print(f"- Mejor kernel: {best_kernel} con C={best_C}, gamma={best_gamma} -> accuracy {best_score_svm:.3f}")
print(f"- Vectores de soporte: {len(model_svm.support_vectors_)} ({len(model_svm.support_vectors_) / len(X_train_scaled) * 100:.1f}% del entrenamiento)")

print(f"\n- **Rendimiento**: accuracy = {acc_svm:.3f}. El modelo acierta aproximadamente el {acc_svm*100:.1f}% de los casos en test.")
print(f"- **Clase 0 (no sobrevive)**: precision = {report['0']['precision']:.3f}, recall = {report['0']['recall']:.3f}. Predice bien a los no supervivientes.")
print(f"- **Clase 1 (sobrevive)**: precision = {report['1']['precision']:.3f}, recall = {report['1']['recall']:.3f}. La precisión es alta, indicando confianza cuando predice supervivencia.")
print(f"- **Matriz de confusión**: TN={tn}, FP={fp}, FN={fn}, TP={tp}. El sesgo es {sesgo}.")
print(f"- **Hiperparámetros optimizados**: kernel={best_kernel}, C={best_C}, gamma={best_gamma}. Estos valores mejoran la accuracy al equilibrar ajuste y generalización.")
print(f"- **Mejora general**: El modelo SVM optimizado logra un accuracy del {acc_svm*100:.1f}% y mantiene buena precisión en la clase de supervivencia.")

# 6. Red Neuronal (Adam)

Resolver el problema de supervivencia del Titanic con un perceptron multicapa usando el dataset preprocesado.

In [ ]:
from sklearn.neural_network import MLPClassifier

## 6.1 Breve explicacion de la red neuronal

Un **perceptrón multicapa** es una red formada por una capa de entrada, una o varias capas ocultas y una capa de salida. Cada neurona calcula una suma ponderada de sus entradas y le aplica una funcion de activacion (por ejemplo `relu` o `tanh`).

La red aprende ajustando los pesos w y los bias b mediante:

1. **Feedforward**: calcula la prediccion propagando las entradas hacia delante.
2. **Backpropagation**: compara la prediccion con la respuesta correcta y propaga el error hacia atras.
3. **Descenso de gradiente**: actualiza los pesos en la direccion que reduce el error.

Vamos a usar `MLPClassifier`.

## 6.2 Entrenamiento y evaluacion

In [ ]:
model_mlp = MLPClassifier(
    hidden_layer_sizes=(10,),
    activation='relu',
    max_iter=500,
    random_state=42
)
model_mlp.fit(X_train_scaled, y_train)

y_pred_mlp = model_mlp.predict(X_test_scaled)

print('Accuracy:', accuracy_score(y_test, y_pred_mlp))
print('Clasification report:')
print(classification_report(y_test, y_pred_mlp))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred_mlp))

## 6.3 Curva de perdida

`MLPClassifier` guarda en `loss_curve_` el valor de la funcion de perdida en cada iteracion del entrenamiento. Solo esta disponible cuando el solver entrena por epocas (`adam` o `sgd`); con `lbfgs` no se rellena porque optimiza en bloque.

In [ ]:
if hasattr(model_mlp, 'loss_curve_'):
    plt.plot(model_mlp.loss_curve_)
    plt.xlabel('Iteracion')
    plt.ylabel('Perdida (loss)')
    plt.title('Evolucion del entrenamiento')
    plt.show()
else:
    print(f"El solver '{model_mlp.solver}' no expone loss_curve_.")

## 6.4 Busqueda simple de hiperparametros

Probamos manualmente varias configuraciones y nos quedamos con la de mayor accuracy.

In [ ]:
configs = [
    {'hidden_layer_sizes': (8,),    'activation': 'relu'},
    {'hidden_layer_sizes': (10,),   'activation': 'relu'},
    {'hidden_layer_sizes': (10, 5), 'activation': 'relu'},
    {'hidden_layer_sizes': (10,),   'activation': 'tanh'},
]

best_score_mlp = 0.0
best_cfg = None

for cfg in configs:
    temp_model = MLPClassifier(**cfg, max_iter=500, random_state=42)
    temp_model.fit(X_train_scaled, y_train)
    score = temp_model.score(X_test_scaled, y_test)
    print(f'hidden_layer_sizes={cfg["hidden_layer_sizes"]}, activation={cfg["activation"]} -> accuracy={score:.4f}')
    if score > best_score_mlp:
        best_score_mlp = score
        best_cfg = cfg

print('\nMejor configuracion:', best_cfg)
print('Mejor accuracy:', best_score_mlp)

## 6.5 Conclusion y explicacion de resultados

In [ ]:
acc_mlp = accuracy_score(y_test, y_pred_mlp)
cm = confusion_matrix(y_test, y_pred_mlp)
report = classification_report(y_test, y_pred_mlp, output_dict=True)

tn, fp, fn, tp = cm.ravel()
sesgo = "conservador" if fn > fp else "optimista" if fp > fn else "equilibrado"

print("Conclusion y explicacion (resumen automatico)")
print(f"- Accuracy: {acc_mlp:.3f}")
print(f"- Clase 0 (no sobrevive): precision={report['0']['precision']:.3f}, recall={report['0']['recall']:.3f}")
print(f"- Clase 1 (sobrevive): precision={report['1']['precision']:.3f}, recall={report['1']['recall']:.3f}")
print(f"- Matriz de confusion: TN={tn}, FP={fp}, FN={fn}, TP={tp} -> sesgo {sesgo}")
print(f"- Mejor configuracion: {best_cfg} con accuracy {best_score_mlp:.3f}")
print(f"- Numero de iteraciones realizadas: {model_mlp.n_iter_}")

# 7. Comparativa global

Tabla resumen de los cuatro modelos sobre el mismo conjunto de test.

In [ ]:
resultados = pd.DataFrame({
    'Modelo': ['Árbol de decisiones', 'KNN', 'SVM', 'Red neuronal (Adam)'],
    'Accuracy': [acc_tree, acc_knn, acc_svm, acc_mlp]
})
resultados = resultados.sort_values('Accuracy', ascending=False).reset_index(drop=True)
print(resultados.to_string(index=False))

mejor = resultados.iloc[0]
print(f"\nMejor modelo según accuracy: {mejor['Modelo']} ({mejor['Accuracy']:.3f})")

## Conclusión final

De los cuatro modelos evaluados sobre el mismo conjunto de test, el de mayor accuracy es el que aparece en primer lugar en la tabla anterior. Los cuatro se mueven en una franja estrecha (en torno al 80–85%), por lo que la diferencia entre ellos es modesta y sus rendimientos son comparables.